# Phase 2 : Expérimentation RAG (Retrieval-Augmented Generation)

Ce notebook démontre l'utilisation de notre système RAG en environnement de développement.
Le RAG permet de pallier les hallucinations des grands modèles de langage en les forçant à se baser sur un contexte documentaire pertinent (nos cours au format PDF).

## Étape 1 : Imports et Configuration
Nous commençons par importer nos modules internes : chargement de documents, base vectorielle FAISS, et notre générateur LLM (Bloomz).

In [ ]:
import os
import sys

os.chdir("..")
sys.path.insert(0, ".")

from src.rag_engine.document_loader import load_pdf_documents, get_chunks
from src.rag_engine.vector_store import VectorStoreManager
from src.generator.llm_generator import RAGGenerator
from src.config import *

## Étape 2 : Chargement du Corpus (PDF)
Nous lisons tous les fichiers PDF situés dans notre dossier `data/corpus/` et nous extrayons le texte brut de chaque page.

In [ ]:
documents = load_pdf_documents(CORPUS_DIR)
print(f"📄 {len(documents)} pages chargées depuis les PDF")

## Étape 3 : Découpage Stratégique (Chunking)
Pour ne pas dépasser la limite de contexte du modèle et pour que la recherche vectorielle soit précise, nous découpons ces pages en petits morceaux (chunks) homogènes, en veillant à ne pas couper les mots en plein milieu.

In [ ]:
chunks = get_chunks(documents)
print(f"✂️ {len(chunks)} chunks créés avec succès")

## Étape 4 : Création de la Base de Données Vectorielle (FAISS)
Chaque chunk de texte est transformé en un vecteur mathématique dense (Embedding) via le modèle `all-MiniLM-L6-v2`. Ces vecteurs sont ensuite stockés dans notre index FAISS pour permettre des recherches sémantiques très rapides.

In [ ]:
manager = VectorStoreManager(EMBEDDING_MODEL_NAME, VECTOR_DB_PATH)
manager.build_and_save_index(chunks)

## Étape 5 : Test de Recherche Sémantique (Retrieval)
Simulons une question utilisateur. Le système va convertir la question en vecteur, et rechercher dans FAISS les chunks les plus similaires sémantiquement.

In [ ]:
query = "Qu'est-ce qu'un système RAG ?"
print(f"🔍 Question posée : {query}\n")

results = manager.search_top_k(query, k=3)
for i, r in enumerate(results, 1):
    print(f"--- Résultat {i} (Source : {r['source']}) ---")
    print(r["text"][:250] + "...\n")

## Étape 6 : Génération Augmentée (Generation)
Nous injectons ces documents pertinents dans le 'Prompt' (l'instruction) de notre LLM local (Bloomz). Le modèle générera une réponse basée **uniquement** sur ces informations.

In [ ]:
generator = RAGGenerator()
response = generator.generate(query, results)
print("\n🤖 Réponse IA (Avec RAG) :\n", response)

## Étape 7 : Comparaison et Démonstration de l'Utilité du RAG
Pour prouver la nécessité de notre système, posons la même question au modèle, mais cette fois-ci **sans lui fournir les documents de cours**.

In [ ]:
response_no_rag = generator.generate_without_rag(query)
print("❌ Réponse IA (SANS RAG) :\n", response_no_rag)
print("\n✅ Rappel Réponse IA (AVEC RAG) :\n", response)